# Bias mitigation with Reweighing (Kamiran & Calders, 2012)

**Reweighing** is a pre-processing method. Each training example gets a weight so that the protected
attribute and the label become independent in the training data. A classifier is then trained with these weights.
The implementation comes from [AIF360](https://github.com/Trusted-AI/AIF360).

Steps for each dataset (Adult, German Credit, COMPAS):
1. Fairness of the original labels: DP_Diff, DIR and consistency.
2. Stratified train/test split.
3. Reweighing on the training data, then train a logistic regression with the weights.
4. Fairness and accuracy of the predictions on the test set.
5. Compare before and after.

In [ ]:
# Installing the packages this notebook needs (only needed once; afterwards you can skip this cell).
%pip install aif360 scikit-learn pandas

In [1]:
import logging                                
logging.getLogger().setLevel(logging.ERROR)   # This hides annoying AIF360's messages about optional packages we don't need to use

import numpy as np                            
import pandas as pd                           
from sklearn.model_selection import train_test_split   
from sklearn.preprocessing import MinMaxScaler          
from sklearn.linear_model import LogisticRegression     
from sklearn.neighbors import NearestNeighbors          
from sklearn.metrics import accuracy_score              
from aif360.datasets import BinaryLabelDataset         
from aif360.algorithms.preprocessing import Reweighing

## Fairness measures
`s` is the protected attribute (1 = privileged group, 0 = unprivileged group) and `y` the outcome (1 = favourable).

* **DP_Diff** = favourable rate of the unprivileged group − favourable rate of the privileged group (ideal 0)
* **DIR** = favourable rate of the unprivileged group / favourable rate of the privileged group (ideal 1)
* **Consistency** = 1 − average difference between each person's outcome and that of their 5 nearest neighbours (ideal 1)

In [ ]:
def fairness_metrics(y, s, X):

    rate_unpriv = y[s == 0].mean()   # share of favourable outcomes (y = 1) in the unprivileged group
    rate_priv = y[s == 1].mean()     # share of favourable outcomes (y = 1) in the privileged group

    # For every person, find the 5 most similar people based on their features.
    neighbours = NearestNeighbors(n_neighbors=5).fit(X).kneighbors(X, return_distance=False)

    return {
        # DP_Diff: difference in favourable rates. 0 = equal, negative = unprivileged group gets fewer favourable outcomes
        "DP_Diff": rate_unpriv - rate_priv,
        # DIR: ratio of favourable rates. where 1 = equal
        "DIR": rate_unpriv / rate_priv,
        # Consistency: y[neighbours].mean(axis=1) is the average outcome of each person's 5 neighbours.
        # We compare it with the person's own outcome, average the differences over everyone, and subtract from 1. Here 1 means similar people always get the same outcome.
        "Consistency": 1 - np.mean(np.abs(y - y[neighbours].mean(axis=1))),
    }


def run(name, X, y, s):
    # Runs the complete pipeline for a dataset and gives a small results table.

    # The features are rescaled to 0-1 first, so that features with big numbers (like income) don't dominate when finding the nearest neighbours.
    before = fairness_metrics(y, s, MinMaxScaler().fit_transform(X))

    
    # 70% of the rows go to training, 30% to testing. stratify=y keeps the same share of favourable labels
    # random_state=8 makes the split the same every time you run the notebook. I usually choose 8 because it's my favourite number, but it can be any number. 
    X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
        X, y, s, test_size=0.3, stratify=y, random_state=8)

    scaler = MinMaxScaler().fit(X_train)   
    X_train = scaler.transform(X_train)    
    X_test = scaler.transform(X_test)      

   
    # AIF360 needs its own data format: a table with the protected attribute "s" and the label "y".
    train = BinaryLabelDataset(df=pd.DataFrame({"s": s_train, "y": y_train}),
                               label_names=["y"],                 
                               protected_attribute_names=["s"])   

    # Telling the Reweighing which value of s is the unprivileged group (0) and which is the privileged group (1).
    reweighing = Reweighing(unprivileged_groups=[{"s": 0}], privileged_groups=[{"s": 1}])

    #This creates one weight for each row in the training set, based on the person's group and label.
    weights = reweighing.fit_transform(train).instance_weights

    # Train a logistic regression. sample_weight makes rows with a high weight count more during training.
    model = LogisticRegression(max_iter=1000).fit(X_train, y_train, sample_weight=weights)

    y_pred = model.predict(X_test)                          # predicted outcome (0/1) for every test person
    after = fairness_metrics(y_pred, s_test, X_test)        # same fairness measures, now on the predictions
    after["Accuracy"] = accuracy_score(y_test, y_pred)      # share of test predictions that match the true label

    # Put "before" and "after" in one table with two rows, labelled with the dataset name.
    return pd.DataFrame([before, after],
                        index=[[name, name], ["Before (original labels)", "After (test predictions)"]])

## Datasets
| Dataset | Protected attribute | Privileged group | Favourable outcome |
|---|---|---|---|
| Adult | sex | male | income > \$50K |
| German Credit | age | older than 25 | good credit |
| COMPAS | race | Caucasian | no reoffence within two years |

The protected attribute is removed from the features, so the model does not use it directly.

In [ ]:
# For every dataset we create three things:
#   X = the features (numbers only), y = the label (1 = favourable), s = the protected attribute (1 = privileged)

#Adult dataset:
# The file has no header row, so we need to give the column names ourselves.
cols = ["age", "workclass", "fnlwgt", "education", "education_num", "marital_status", "occupation",
        "relationship", "race", "sex", "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"]
adult = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
                    names=cols,                 
                    skipinitialspace=True,      
                    na_values="?").dropna()     # missing values are written as "?": this marks them as missing, then drop those rows
adult_y = (adult["income"] == ">50K").astype(int).values   # 1 if income > 50K (favourable), else 0
adult_s = (adult["sex"] == "Male").astype(int).values      # 1 if male (privileged), else 0

# get_dummies turns text columns into 0/1 columns (one-hot encoding), e.g. workclass -> workclass_Private, ...
# .values turns the table into a plain numpy array of numbers.
adult_X = pd.get_dummies(adult.drop(columns=["income", "sex"])).values

#German credit dataset:
# Values are separated by spaces and there is no header row, so the columns are numbered 0-20.
german = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data",
                     sep=" ", header=None)
german_y = (german[20] == 1).astype(int).values   # column 20 is the label: 1 = good credit (favourable), 2 = bad
german_s = (german[12] > 25).astype(int).values   # column 12 is age: older than 25 = privileged
german_X = pd.get_dummies(german.drop(columns=[12, 20])).values   # features: everything except age and the label

#COMPAS dataset:
compas = pd.read_csv("https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv")
compas_y = (compas["two_year_recid"] == 0).astype(int).values   # 1 = did NOT reoffend within two years (favourable)
compas_s = (compas["race"] == "Caucasian").astype(int).values   # 1 = Caucasian (privileged)
# Features: a few basic columns about the person and their criminal history (race is left out).
compas_X = pd.get_dummies(compas[["sex", "age", "c_charge_degree", "priors_count",
                                  "juv_fel_count", "juv_misd_count", "juv_other_count"]]).values

## Results

In [5]:
# Run the full pipeline on all three datasets and stack the three result tables on top of each other.
results = pd.concat([
    run("Adult", adult_X, adult_y, adult_s),
    run("German", german_X, german_y, german_s),
    run("COMPAS", compas_X, compas_y, compas_s),
])
results.round(3)   # show the table, rounded to 3 decimals

DP_Diff    DIR  Consistency  Accuracy
Adult  Before (original labels)   -0.200  0.362        0.829       NaN
       After (test predictions)   -0.091  0.580        0.947     0.840
German Before (original labels)   -0.149  0.795        0.743       NaN
       After (test predictions)   -0.056  0.925        0.799     0.737
COMPAS Before (original labels)   -0.086  0.858        0.644       NaN
       After (test predictions)   -0.161  0.789        0.986     0.674